In [4]:
import re
from better_profanity import profanity

class WordFilter:
    def __init__(self,
    min_length=3,
    max_length=15, 
    lang_model="en_core_web_trf"):

        self.min_length = min_length
        self.max_length = max_length
        profanity.load_censor_words()
        self.nlp = spacy.load(lang_model, disable=["parser"])
    
    def is_valid_length(self,word):
        return self.min_length <= len(word) <= self.max_length
    
    def is_clean(self, word):
        return not profanity.contains_profanity(word)

    def filter(self, list_of_words):
        result = []
        doc = self.nlp(" ".join(list_of_words))
        for token in doc:
            lemma = token.lemma_.lower()
            #print(f'{lemma.isalpha()} {self.is_valid_length(lemma)} {self.is_clean(lemma)}')
            if lemma.isalpha() and \
            self.is_valid_length(lemma) \
            and self.is_clean(lemma) \
            and token.ent_type_ == '':
                if lemma not in result:
                    result.append(lemma)
        return result
        
    
word_filter = WordFilter(min_length=3,max_length=15)
word_filter.filter(["hello","pinochet","james","world", ])

['hello']

In [ ]:
!pip install spacy-curated-transformers
!python -m spacy download en_core_web_trf

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [1]:
import spacy_transformers
import spacy
nlp = spacy.load("en_core_web_trf")

/Users/eriq/Desktop/word.golf/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from wordfreq import top_n_list
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import csv

word_filter = WordFilter(min_length=3,max_length=15)

def generate_embeddings(filename,vocab_size=40000, 
    top_k=120,
    model="sentence-transformers/all-MiniLM-L6-v2", 
    word_filter=word_filter):

    vocab = top_n_list('en', vocab_size)
    vocab = word_filter.filter(vocab)
    
    model = SentenceTransformer(model)
    embeddings = model.encode(vocab, batch_size=512, show_progress_bar=True)
    normed = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    # save embeddings as csv
    with open(f"embed_{filename}", "w") as f:
        writer = csv.writer(f)
        for word, vector in zip(vocab, normed):
            writer.writerow([word] + vector.tolist())
    cos_sim = cosine_similarity(normed)

    neighbors = []

    for i in tqdm(range(len(vocab))):
        sim_scores = cos_sim[i]
        # Exclude self-match by setting it to -inf
        sim_scores[i] = -np.inf
        top_indices = np.argpartition(sim_scores, -top_k)[-top_k:]
        top_sorted = top_indices[np.argsort(-sim_scores[top_indices])]
        neighbor_words = [vocab[j] for j in top_sorted]
        neighbors.append([vocab[i]] + neighbor_words)

    df = pd.DataFrame(neighbors)
    df.to_csv(filename, index=False, header=False)

generate_embeddings("sbert_neighbors_top100_ner_web_trf.csv", 40000, 100)

100%|██████████| 26505/26505 [00:05<00:00, 4838.39it/s]


In [ ]:
df = pd.read_csv("../data/sbert_word_embeddings.csv")
def vector_for_word(word):
    # Check if the word is in the DataFrame
    if word in df.iloc[:, 0].values:
        return df[df.iloc[:, 0] == word].iloc[0, 1:].values
    else:
        return None
        
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    if norm_a == 0 or norm_b == 0:
        return 0.0 
    return dot_product / (norm_a * norm_b)

v1 = vector_for_word("kite")
v2 = vector_for_word("dog")
cosine_similarity(v1, v2)

0.1886469751209768